In [1]:
import requests
import pandas as pd
import io

In [2]:
def get_tuva_value_sets(url):
    response = requests.get(url)
    response.raise_for_status()

    csv_bytes = io.BytesIO(response.content)
    
    try:
        df = pd.read_csv(csv_bytes, compression='gzip', header=None)
    except Exception as e:
        print(f"Failed to read as gzipped CSV: {e}. Trying without gzip compression.")
        csv_bytes.seek(0)
        df = pd.read_csv(csv_bytes, header=None)
    
    return df

def format_icd_code(code):
    if isinstance(code, str) and len(code) > 3:
        return code[:3] + '.' + code[3:]
    return code

urls = {
    'ed_classification_categories': 'https://tuva-public-resources.s3.amazonaws.com/versioned_value_sets/latest/ed_classification_categories.csv_0_0_0.csv.gz',
    'icd_10_cm_to_ccs': 'https://tuva-public-resources.s3.amazonaws.com/versioned_value_sets/latest/icd_10_cm_to_ccs.csv_0_0_0.csv.gz',
    'johnston_icd10': 'https://tuva-public-resources.s3.amazonaws.com/versioned_value_sets/latest/johnston_icd10.csv_0_0_0.csv.gz',
    'johnston_icd9': 'https://tuva-public-resources.s3.amazonaws.com/versioned_value_sets/latest/johnston_icd9.csv_0_0_0.csv.gz',
    'cms_chronic_conditions_hierarchy': 'https://tuva-public-resources.s3.amazonaws.com/versioned_value_sets/latest/cms_chronic_conditions_hierarchy.csv_0_0_0.csv.gz'
}

header_info = {
    'ed_classification_categories': [
        'classification', 'classification_name', 'classification_order', 'classification_column'
    ],
    'icd_10_cm_to_ccs': [
        'icd_10_cm', 'description', 'ccs_diagnosis_category', 'ccs_description'
    ],
    'johnston_icd10': [
        'icd10', 'edcnnpa', 'edcnpa', 'epct', 'noner', 'injury', 'psych', 'alcohol', 'drug'
    ],
    'johnston_icd9': [
        'icd9', 'edcnnpa', 'edcnpa', 'epct', 'noner', 'injury', 'psych', 'alcohol', 'drug'
    ],
    'cms_chronic_conditions_hierarchy': [
        'condition_id', 'condition', 'condition_column_name', 'chronic_condition_type',
        'condition_category', 'additional_logic', 'claims_qualification', 'inclusion_type',
        'code_system', 'code'
    ]
}

In [3]:
dataframes = {}

for name, url in urls.items():
    print(f"Downloading and loading {name}...")
    df = get_tuva_value_sets(url)
    df.columns = header_info[name]
    dataframes[name] = df
    print(f"{name} loaded with shape {df.shape}")

ed_classification_categories loaded with shape (9, 4)
Failed to read as gzipped CSV: Not a gzipped file (b'"A'). Trying without gzip compression.
icd_10_cm_to_ccs loaded with shape (72776, 4)
johnston_icd10 loaded with shape (47132, 9)
johnston_icd9 loaded with shape (7443, 9)
Failed to read as gzipped CSV: Not a gzipped file (b'1,'). Trying without gzip compression.
cms_chronic_conditions_hierarchy loaded with shape (6532, 10)


In [4]:
dataframes['icd_10_cm_to_ccs']['icd_10_cm'] = dataframes['icd_10_cm_to_ccs']['icd_10_cm'].apply(format_icd_code)
dataframes['johnston_icd10']['icd10'] = dataframes['johnston_icd10']['icd10'].apply(format_icd_code)
dataframes['johnston_icd9']['icd9'] = dataframes['johnston_icd9']['icd9'].apply(format_icd_code)
dataframes['cms_chronic_conditions_hierarchy'].loc[
    dataframes['cms_chronic_conditions_hierarchy']['code_system'] == 'ICD-10-CM', 'code'
] = dataframes['cms_chronic_conditions_hierarchy'].loc[
    dataframes['cms_chronic_conditions_hierarchy']['code_system'] == 'ICD-10-CM', 'code'
].apply(format_icd_code)

In [5]:
dataframes['ed_classification_categories']

,classification,classification_name,classification_order,classification_column
0,edcnnpa,"Emergent, ED Care Needed, Not Preventable/Avoi...",4,emergent_ed_not_preventable
1,noner,Non-Emergent,1,non_emergent
2,injury,Injury,5,injury
3,epct,"Emergent, Primary Care Treatable",2,emergent_primary_care
4,edcnpa,"Emergent, ED Care Needed, Preventable/Avoidable",3,emergent_ed_preventable
5,psych,Mental Health Related,6,mental_health
6,alcohol,Alcohol Related,7,alcohol
7,drug,Drug Related (excluding alcohol),8,drug
8,unclassified,"Not in a Special Category, and Not Classified",9,unclassified


In [6]:
dataframes['icd_10_cm_to_ccs']

,icd_10_cm,description,ccs_diagnosis_category,ccs_description
0,A00.0,"Cholera due to Vibrio cholerae 01, biovar chol...",135,Intestinal infection
1,A00.1,"Cholera due to Vibrio cholerae 01, biovar eltor",135,Intestinal infection
2,A00.9,"Cholera, unspecified",135,Intestinal infection
3,A01.00,"Typhoid fever, unspecified",135,Intestinal infection
4,A01.01,Typhoid meningitis,76,Meningitis (except that caused by tuberculosis...
...,...,...,...,...
72771,Z99.2,Dependence on renal dialysis,158,Chronic kidney disease
72772,Z99.3,Dependence on wheelchair,259,Residual codes; unclassified
72773,Z99.81,Dependence on supplemental oxygen,131,Respiratory failure; insufficiency; arrest (ad...
72774,Z99.89,Dependence on other enabling machines and devices,259,Residual codes; unclassified


In [7]:
dataframes['johnston_icd10']

,icd10,edcnnpa,edcnpa,epct,noner,injury,psych,alcohol,drug
0,A02.0,0.000000,0.0,1.000000,0.000000,0,0,0,0
1,A03.9,0.171429,0.0,0.457143,0.371429,0,0,0,0
2,A04.9,0.171429,0.0,0.457143,0.371429,0,0,0,0
3,A05.9,0.171429,0.0,0.457143,0.371429,0,0,0,0
4,A08.4,0.171429,0.0,0.457143,0.371429,0,0,0,0
...,...,...,...,...,...,...,...,...,...
47127,Z99.11,1.000000,0.0,0.000000,0.000000,0,0,0,0
47128,Z99.12,1.000000,0.0,0.000000,0.000000,0,0,0,0
47129,Z99.2,0.941176,0.0,0.058824,0.000000,0,0,0,0
47130,Z99.3,1.000000,0.0,0.000000,0.000000,0,0,0,0


In [8]:
dataframes['johnston_icd9']

,icd9,edcnnpa,edcnpa,epct,noner,injury,psych,alcohol,drug
0,003.0,0.000000,0.0,1.000000,0.000000,0,0,0,0
1,004.9,0.171429,0.0,0.457143,0.371429,0,0,0,0
2,005.9,0.171429,0.0,0.457143,0.371429,0,0,0,0
3,008.5,0.171429,0.0,0.457143,0.371429,0,0,0,0
4,008.8,0.171429,0.0,0.457143,0.371429,0,0,0,0
...,...,...,...,...,...,...,...,...,...
7438,V91.29,0.052632,0.0,0.368421,0.578947,0,0,0,0
7439,V91.90,0.052632,0.0,0.368421,0.578947,0,0,0,0
7440,V91.91,0.052632,0.0,0.368421,0.578947,0,0,0,0
7441,V91.92,0.052632,0.0,0.368421,0.578947,0,0,0,0


In [9]:
dataframes['cms_chronic_conditions_hierarchy']

,condition_id,condition,condition_column_name,chronic_condition_type,condition_category,additional_logic,claims_qualification,inclusion_type,code_system,code
0,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I23.0
1,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I23.1
2,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I23.2
3,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I23.3
4,1,Acute Myocardial Infarction,acute_myocardial_infarction,Common,Cardiovascular Disease,NaN,At least 1 inpatient claim with DX codes,Include,ICD-10-CM,I23.4
...,...,...,...,...,...,...,...,...,...,...
6527,75,Viral Hepatitis (general),viral_hepatitis_general,Other chronic or potentially disabling conditions,Viral Hepatitis,NaN,At least 1 inpatient claim OR 2 non- inpatient...,Include,ICD-10-CM,B19.9
6528,75,Viral Hepatitis (general),viral_hepatitis_general,Other chronic or potentially disabling conditions,Viral Hepatitis,NaN,At least 1 inpatient claim OR 2 non- inpatient...,Include,ICD-10-CM,Z22.50
6529,75,Viral Hepatitis (general),viral_hepatitis_general,Other chronic or potentially disabling conditions,Viral Hepatitis,NaN,At least 1 inpatient claim OR 2 non- inpatient...,Include,ICD-10-CM,Z22.51
6530,75,Viral Hepatitis (general),viral_hepatitis_general,Other chronic or potentially disabling conditions,Viral Hepatitis,NaN,At least 1 inpatient claim OR 2 non- inpatient...,Include,ICD-10-CM,Z22.52


In [10]:
# show all the unique values in the code_system column
unique_code_systems = dataframes['cms_chronic_conditions_hierarchy']['code_system'].unique()
print("Unique code systems in cms_chronic_conditions_hierarchy:")
for code_system in unique_code_systems:
    print(code_system)

Unique code systems in cms_chronic_conditions_hierarchy:
ICD-10-CM
ICD-10-PCS
HCC
MS-DRG
HCPCS
NDC


In [11]:
rows = dataframes['cms_chronic_conditions_hierarchy'][dataframes['cms_chronic_conditions_hierarchy']['code_system'] == 'ICD-10-PCS']
rows

,condition_id,condition,condition_column_name,chronic_condition_type,condition_category,additional_logic,claims_qualification,inclusion_type,code_system,code
119,3,Alcohol Use Disorders,alcohol_use_disorders,Other chronic or potentially disabling conditions,Substance Abuse,NaN,At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ2ZZZZ
120,3,Alcohol Use Disorders,alcohol_use_disorders,Other chronic or potentially disabling conditions,Substance Abuse,NaN,At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ30ZZZ
121,3,Alcohol Use Disorders,alcohol_use_disorders,Other chronic or potentially disabling conditions,Substance Abuse,NaN,At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ31ZZZ
122,3,Alcohol Use Disorders,alcohol_use_disorders,Other chronic or potentially disabling conditions,Substance Abuse,NaN,At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ32ZZZ
123,3,Alcohol Use Disorders,alcohol_use_disorders,Other chronic or potentially disabling conditions,Substance Abuse,NaN,At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ33ZZZ
...,...,...,...,...,...,...,...,...,...,...
3974,55,Opioid Use Disorder (OUD),opioid_use_disorder_oud,Other chronic or potentially disabling conditions,Substance Abuse,"Exclude beneficiaries with NDC for Naltrexone,...",At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ86ZZZ
3975,55,Opioid Use Disorder (OUD),opioid_use_disorder_oud,Other chronic or potentially disabling conditions,Substance Abuse,"Exclude beneficiaries with NDC for Naltrexone,...",At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ91ZZZ
3976,55,Opioid Use Disorder (OUD),opioid_use_disorder_oud,Other chronic or potentially disabling conditions,Substance Abuse,"Exclude beneficiaries with NDC for Naltrexone,...",At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ94ZZZ
3977,55,Opioid Use Disorder (OUD),opioid_use_disorder_oud,Other chronic or potentially disabling conditions,Substance Abuse,"Exclude beneficiaries with NDC for Naltrexone,...",At least 1 inpatient claim OR 2 other non-drug...,Include,ICD-10-PCS,HZ95ZZZ
